# Regressão Logística

In [5]:

import joblib
from multiprocessing import cpu_count
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import os
from scipy.stats import uniform, loguniform
from numpy import linspace
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.metrics import f1_score

# 1 ABERTURA DOS DADOS

In [6]:
# Caminho
data_dir = "C:/Users/Elaine/Desktop/TCC_acidentes/models/"

# Carregar o preprocess
preprocess = joblib.load('preprocess.pkl')

# Carregar os splits
X_train, X_test, y_train, y_test = joblib.load('data_splits.pkl')

X_train.info()
X_train.head()

<class 'pandas.DataFrame'>
Index: 140662 entries, 6908 to 90899
Data columns (total 24 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   uf                      140662 non-null  str    
 1   br                      140285 non-null  float64
 2   km                      140285 non-null  float64
 3   dia_semana              140662 non-null  str    
 4   fase_dia                140662 non-null  str    
 5   sentido_via             140662 non-null  str    
 6   condicao_metereologica  140662 non-null  str    
 7   tipo_pista              140662 non-null  str    
 8   uso_solo                140662 non-null  str    
 9   tipo_veiculo            135943 non-null  str    
 10  ano                     140662 non-null  int64  
 11  frota                   139581 non-null  float64
 12  Aclive                  140662 non-null  int64  
 13  Curva                   140662 non-null  int64  
 14  Declive                 140662 non

,uf,br,km,dia_semana,fase_dia,sentido_via,condicao_metereologica,tipo_pista,uso_solo,tipo_veiculo,...,Declive,Desvio Temporário,Em Obras,Interseção de Vias,Ponte,Reta,Retorno Regulamentado,Rotatória,Túnel,Viaduto
6908,MG,116.0,425.0,domingo,Pleno dia,Crescente,Nublado,Simples,Sim,Automóvel,...,0,0,0,0,0,1,0,0,0,0
114351,BA,101.0,837.0,quarta-feira,Pleno dia,Crescente,Chuva,Simples,Não,Caminhão-trator,...,1,0,0,0,0,0,0,0,0,0
168709,PA,230.0,92.0,sexta-feira,Pleno dia,Decrescente,Céu Claro,Simples,Não,Caminhão-trator,...,0,0,0,0,0,1,0,0,0,0
35822,PE,232.0,405.0,quinta-feira,Plena Noite,Crescente,Céu Claro,Simples,Não,Caminhão,...,0,0,0,0,0,0,0,0,0,0
148029,PR,116.0,6.0,domingo,Plena Noite,Crescente,Nublado,Dupla,Sim,Automóvel,...,0,0,0,0,0,1,0,0,0,0


# 2 OTMIZAÇÃO DO MODELO

In [7]:
# Pipeline
pipeline_lr = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=300))
])

# Grid
param_grid_lr = {
    'clf__solver': ['saga'],                
    'clf__penalty': ['elasticnet'],         
    'clf__C': loguniform(1e-2, 1e2),
    'clf__l1_ratio': uniform(0, 1),        
    'clf__class_weight': [None, 'balanced']
}

# Busca
random_lr = RandomizedSearchCV(
    estimator=pipeline_lr,
    param_distributions=param_grid_lr,  
    n_iter=100,                           
    scoring='f1',
    cv=5,
    n_jobs=cpu_count() // 2,
    random_state=42,
    verbose=1
)


In [ ]:
# Fit
random_lr.fit(X_train, y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [6]:
# Resultados
y_pred_lr = random_lr.predict(X_test)
print(f"Melhor F1 na Validação Cruzada: {random_lr.best_score_:.4f}")
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

Melhor F1 na Validação Cruzada: 0.3911
              precision    recall  f1-score   support

           0       0.92      0.66      0.77     19305
           1       0.28      0.70      0.40      3603

    accuracy                           0.67     22908
   macro avg       0.60      0.68      0.58     22908
weighted avg       0.82      0.67      0.71     22908

[[12769  6536]
 [ 1088  2515]]


# 3 DIAGNÓSTICO

## 3.1 Métricas Básicas

In [27]:
# Resultados
y_pred_lr = random_lr.predict(X_test)
print(f"Melhor F1 na Validação Cruzada: {random_lr.best_score_:.4f}")
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

Melhor F1 na Validação Cruzada: 0.3908
              precision    recall  f1-score   support

           0       0.92      0.66      0.77     19305
           1       0.28      0.70      0.40      3603

    accuracy                           0.67     22908
   macro avg       0.60      0.68      0.58     22908
weighted avg       0.82      0.67      0.71     22908

[[12764  6541]
 [ 1095  2508]]


## 3.2 Mudança de Limiar do Predict

In [28]:
y_proba = random_lr.predict_proba(X_test)[:, 1]

# 2. Teste vários limiares
thresholds = linspace(0, 0.9, 25)

print("Limiar | F1-Score | Recall | Precision")
print("---------------------------------------")
for th in thresholds:
    y_pred_th = (y_proba >= th).astype(int)
    f1 = f1_score(y_test, y_pred_th)
    # Calculando recall e precision manualmente para printar
    rec = sum((y_pred_th == 1) & (y_test == 1)) / sum(y_test == 1)
    prec = sum((y_pred_th == 1) & (y_test == 1)) / sum(y_pred_th == 1)
    
    print(f" {th:.2f}  |  {f1:.4f}  |  {rec:.4f} |  {prec:.4f}")

Limiar | F1-Score | Recall | Precision
---------------------------------------
 0.00  |  0.2718  |  1.0000 |  0.1573
 0.04  |  0.2945  |  1.0000 |  0.1726
 0.07  |  0.2948  |  1.0000 |  0.1729
 0.11  |  0.2956  |  0.9997 |  0.1734
 0.15  |  0.2982  |  0.9986 |  0.1753
 0.19  |  0.3032  |  0.9947 |  0.1789
 0.22  |  0.3095  |  0.9831 |  0.1837
 0.26  |  0.3193  |  0.9703 |  0.1911
 0.30  |  0.3292  |  0.9470 |  0.1992
 0.34  |  0.3434  |  0.9245 |  0.2109
 0.38  |  0.3547  |  0.8845 |  0.2218
 0.41  |  0.3668  |  0.8354 |  0.2350
 0.45  |  0.3782  |  0.7724 |  0.2504
 0.49  |  0.3899  |  0.7133 |  0.2682
 0.53  |  0.4052  |  0.6520 |  0.2939
 0.56  |  0.4057  |  0.5748 |  0.3135
 0.60  |  0.4079  |  0.5046 |  0.3422
 0.64  |  0.3921  |  0.4230 |  0.3654
 0.67  |  0.3779  |  0.3639 |  0.3931
 0.71  |  0.3486  |  0.2992 |  0.4175
 0.75  |  0.3164  |  0.2417 |  0.4579
 0.79  |  0.2727  |  0.1868 |  0.5049
 0.82  |  0.1943  |  0.1185 |  0.5391
 0.86  |  0.1081  |  0.0594 |  0.5994
 0.90  | 